# Colab Dense Embedding & Indexing - LegalRAG

Notebook này dùng để **tạo Dense Vector Index (Embedding)** cho dữ liệu pháp luật lao động (Bộ luật Lao động 2019) sử dụng GPU (Tesla T4) trên Google Colab.

## Quy trình thực hiện:
1. **Runtime**: Đảm bảo đã chọn **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).
2. **Cài đặt**: Clone mã nguồn LegalRAG và cài đặt thư viện (`pip install -e .`).
3. **Tải dữ liệu Chunk**: Tải file chunk (`.jsonl`) lên Colab.
4. **Build Index**: Chạy CLI `python -m legal_rag.cli.main index dense` để embed tập chunk.
5. **Kiểm tra**: Load index, kiểm tra shape embedding và chạy thử câu hỏi mẫu.
6. **Tải kết quả**: Tải file index `.pkl` về máy.

In [ ]:
# 1. Kiểm tra GPU và môi trường PyTorch CUDA
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Clone repository LegalRAG và cài đặt package
!rm -rf /content/LegalRAG
!git clone https://github.com/ngminhduc129/LegalRAG.git /content/LegalRAG
%cd /content/LegalRAG
!pip install -e . --quiet

# Thêm thư mục src vào sys.path để kernel Python nhận diện module ngay lập tức
import sys
if "/content/LegalRAG/src" not in sys.path:
    sys.path.insert(0, "/content/LegalRAG/src")

In [ ]:
# 3. Chuẩn bị file Chunks dữ liệu
# File chunks (.jsonl) nằm ở local: data/chunks/legal_aware_run06/labor_code_2019_legal_aware.jsonl (~800 KB)
import os
import shutil
from pathlib import Path
from google.colab import files

chunk_dir = Path("data/chunks/legal_aware_run06")
chunk_dir.mkdir(parents=True, exist_ok=True)
input_chunks = chunk_dir / "labor_code_2019_legal_aware.jsonl"

if not input_chunks.exists():
    print("Vui lòng chọn file 'labor_code_2019_legal_aware.jsonl' từ máy tính để tải lên Colab:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith(".jsonl"):
            shutil.move(filename, str(input_chunks))
            print(f"-> Đã lưu file chunk vào: {input_chunks}")
            break
else:
    print(f"File chunk đã tồn tại tại: {input_chunks}")

print(f"Kích thước file chunk: {input_chunks.stat().st_size / 1024:.2f} KB")

In [ ]:
# 4. Build Dense Index bằng sentence-transformers trên GPU
output_dir = Path("data/indexes/dense")
output_dir.mkdir(parents=True, exist_ok=True)
output_index = output_dir / "dense_legal_aware_run06.pkl"

print(f"Đang tạo Dense Index cho {input_chunks}...")
!python -m legal_rag.cli.main index dense {input_chunks} \
    --output {output_index} \
    --model sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 \
    --batch-size 32 \
    --max-length 256

print(f"\n✓ Hoàn tất build index tại: {output_index}")

In [ ]:
# 5. Kiểm tra Index và chạy thử Semantic Search với câu hỏi pháp lý mẫu
import sys
from pathlib import Path
if "/content/LegalRAG/src" not in sys.path:
    sys.path.insert(0, "/content/LegalRAG/src")

from legal_rag.retrieval.dense import DenseIndex

index = DenseIndex.load(Path(output_index))
print(f"Số lượng vectors: {len(index.documents)}")
print(f"Kích thước ma trận embeddings: {index.embeddings.shape}")

# Chạy thử câu hỏi mẫu
query = "Người lao động có quyền đơn phương chấm dứt hợp đồng lao động khi nào?"
print(f"\n--- TEST QUERY: {query} ---\n")
results = index.search(query, k=3)
for rank, (doc_id, score, doc) in enumerate(results, start=1):
    print(f"[Top {rank}] Score: {score:.4f} | ID: {doc_id}")
    print(f"  Văn bản: Điều {doc.get('article', 'N/A')} - {doc.get('article_title', '')}")
    print(f"  Nội dung: {doc.get('content', '')[:180]}...\n")

In [ ]:
# 6. Tải file index về máy local
from google.colab import files

print(f"Đang tải file index {output_index} về máy...")
files.download(str(output_index))